In [ ]:
%pip install ase==3.22.0 -q

In [ ]:
import sys, platform, numpy as np
print('python:', sys.version)
print('numpy:', np.__version__)
print('platform:', platform.platform())

In [ ]:
import numpy as np
from ase.calculators.calculator import Calculator, all_changes


class Gupta(Calculator):
    implemented_properties = ["energy", "forces"]

    def __init__(self, parameters, cutoff=None, debug=False, **kwargs):
        Calculator.__init__(self, **kwargs)
        if len(parameters) != 1:
            raise NotImplementedError("Only single-element Gupta is implemented.")
        (self.symbol, (self.p, self.q, self.A, self.xi, self.r0)), = parameters.items()
        self.cutoff = cutoff

    def calculate(self, atoms=None, properties=("energy",), system_changes=all_changes):
        Calculator.calculate(self, atoms, properties, system_changes)
        if any(atoms.pbc):
            raise NotImplementedError("Periodic systems are not supported.")
        if set(atoms.get_chemical_symbols()) != {self.symbol}:
            raise ValueError("All atoms must be " + self.symbol)
        p, q, A, xi, r0 = self.p, self.q, self.A, self.xi, self.r0
        pos = atoms.positions
        n = len(pos)
        vec = pos[:, None, :] - pos[None, :, :]
        r = np.linalg.norm(vec, axis=2)
        np.fill_diagonal(r, np.inf)
        mask = np.isfinite(r)
        if self.cutoff is not None:
            mask &= r < self.cutoff
        d = np.where(mask, r / r0 - 1.0, 0.0)
        rep = np.where(mask, A * np.exp(-p * d), 0.0)
        att = np.where(mask, xi**2 * np.exp(-2.0 * q * d), 0.0)
        rho = att.sum(axis=1)
        sqrt_rho = np.sqrt(rho)
        energy = rep.sum() - sqrt_rho.sum()
        inv = np.where(sqrt_rho > 0, 0.5 / sqrt_rho, 0.0)
        dE_dr = 2.0 * (-p / r0) * rep + (inv[:, None] + inv[None, :]) * (2.0 * q / r0) * att
        with np.errstate(invalid="ignore", divide="ignore"):
            unit = np.where(mask[:, :, None], vec / r[:, :, None], 0.0)
        forces = -(dE_dr[:, :, None] * unit).sum(axis=1)
        self.results = {"energy": energy, "forces": forces}


def Minimisation_Function(cluster, collection, cluster_name):
    import time
    from ase.optimize import FIRE
    cluster.pbc = False
    r0 = 4.09 / (2.0 ** 0.5)
    Gupta_parameters = {'Ag': [10.85, 3.18, 0.1031, 1.190, r0]}
    cluster.calc = Gupta(Gupta_parameters, cutoff=1000, debug=False)
    dyn = FIRE(cluster, logfile=None)
    startTime = time.time(); converged = False
    try:
        dyn.run(fmax=0.01, steps=5000)
        converged = dyn.converged()
    except Exception:
        print('Local Optimiser Failed for some reason.')
    endTime = time.time()
    Info = {'steps': dyn.get_number_of_steps(), 'time_s': endTime - startTime}
    return cluster, converged, Info

In [ ]:
import time
import numpy as np
from ase.cluster import Icosahedron

rng = np.random.default_rng(0)
for shells in (4, 5, 6):
    cluster = Icosahedron('Ag', shells, latticeconstant=4.09)
    cluster.positions += rng.normal(scale=0.4, size=cluster.positions.shape)
    n = len(cluster)
    t0 = time.time()
    cluster, converged, info = Minimisation_Function(cluster, None, f'ico-{shells}')
    wall = time.time() - t0
    print(f'shells={shells} atoms={n} converged={converged} wall_s={wall:.2f} info={info}')